## VADER

VADER (Valence Aware Dictionary and sEntiment Reasoner) es un analizador de sentimientos basado en un diccionario. Para cada texto devuelve cuatro scores:

· pos — proporción del texto con carga positiva

· neg — proporción del texto con carga negativa

· neu — proporción del texto neutra

· compound — score global de -1 (muy negativo) a +1 (muy positivo)


**Cómo funciona internamente**
No usa una red neuronal como Hartmann. Tiene un diccionario de miles de palabras con valores de sentimiento asignados manualmente por humanos. Por ejemplo:

"love" → +3.2

"hate" → -3.4

"good" → +1.9


Suma los valores de todas las palabras del texto y normaliza el resultado entre -1 y +1.

VADER tiene un diccionario (léxico) donde cada palabra tiene un puntaje de polaridad predefinido — por ejemplo, "good" = +1.9, "terrible" = -2.5, "okay" = +0.9. El proceso es:

· Recorre todo el texto palabra por palabra (sin importar cuán largo sea) y busca cada palabra en ese léxico.

· Ajusta el puntaje según reglas de contexto local — no analiza la frase completa con "comprensión", sino con reglas mecánicas: si detecta una negación cerca ("not good"), invierte o reduce el puntaje; si hay un intensificador ("very good", "extremely good"), lo aumenta; si hay mayúsculas o signos de exclamación ("GREAT!!!"), también lo aumenta.

· Suma todos esos puntajes ajustados de todas las palabras del texto completo, dando un número (sum).

· Normaliza esa suma con esta fórmula para que quede entre -1 y +1:

Todo esto pasa en una sola pasada sobre el texto entero — no hay chunks, no hay media, no hay ponderación por nada. Es literalmente: "suma la carga emocional de cada palabra del texto completo, y normaliza ese total a una escala de -1 a +1."

In [4]:
import pickle
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Cargar dataset
with open("Dataset_final.pkl", "rb") as f:
    df = pickle.load(f)

peliculas_excluidas = ['Talk to Her', 'Anatomy of a Fall']
df = df[~df['Title'].isin(peliculas_excluidas)].reset_index(drop=True)

# Cargar VADER
analyzer = SentimentIntensityAnalyzer()

print(f"Películas cargadas: {len(df)}")

# Test rápido
test = analyzer.polarity_scores("I love you, you are the most beautiful thing I have ever seen")
print(f"\nTest VADER: {test}")

Películas cargadas: 75

Test VADER: {'neg': 0.0, 'neu': 0.567, 'pos': 0.433, 'compound': 0.8553}


In [5]:
frases = [
    "nigger",
    "I will kill you",
    "fuck you dumbass"
]

for frase in frases:
    scores = analyzer.polarity_scores(frase)
    print(f"'{frase}' → {scores}")

'nigger' → {'neg': 1.0, 'neu': 0.0, 'pos': 0.0, 'compound': -0.6486}
'I will kill you' → {'neg': 0.61, 'neu': 0.39, 'pos': 0.0, 'compound': -0.6908}
'fuck you dumbass' → {'neg': 0.877, 'neu': 0.123, 'pos': 0.0, 'compound': -0.7964}


In [ ]:
%%time

import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# ---------- 0. Cargar el dataset ----------
df = pd.read_pickle('Dataset_final_NLP.pkl')

# ---------- 0.1 Excluir películas no válidas para análisis en inglés ----------
# peliculas_excluidas = ['Talk to Her', 'Anatomy of a Fall']
# df = df[~df['Title'].isin(peliculas_excluidas)].reset_index(drop=True)

# ---------- 1. Configurar VADER ----------
analyzer = SentimentIntensityAnalyzer()

# ---------- 2. Función de análisis por personaje ----------

def analyze_vader(texto):
    if not texto or not isinstance(texto, str) or texto.strip() == "":
        return None, None, None, None, None

    scores = analyzer.polarity_scores(texto)
    compound = scores['compound']

    if compound >= 0.05:
        label = 'POSITIVE'
    elif compound <= -0.05:
        label = 'NEGATIVE'
    else:
        label = 'NEUTRAL'

    return scores['pos'], scores['neu'], scores['neg'], compound, label

# ---------- 3. Aplicar a cada personaje dentro de un Script_Dict ----------

def analyze_script_dict_vader(script_dict):
    pos_dict, neu_dict, neg_dict, compound_dict, label_dict = {}, {}, {}, {}, {}
    for personaje, texto in script_dict.items():
        pos, neu, neg, compound, label = analyze_vader(texto)
        pos_dict[personaje] = pos
        neu_dict[personaje] = neu
        neg_dict[personaje] = neg
        compound_dict[personaje] = compound
        label_dict[personaje] = label
    return pos_dict, neu_dict, neg_dict, compound_dict, label_dict

# ---------- 4. Aplicar a todo el dataset ----------

resultados = df['Script_Dict'].apply(analyze_script_dict_vader)

df['VaderSentiment_Pos'] = resultados.apply(lambda x: x[0])
df['VaderSentiment_Neu'] = resultados.apply(lambda x: x[1])
df['VaderSentiment_Neg'] = resultados.apply(lambda x: x[2])
df['VaderSentiment_Compound'] = resultados.apply(lambda x: x[3])
df['VaderSentiment_Label'] = resultados.apply(lambda x: x[4])

# ---------- 5. Guardar el dataset actualizado ----------

df.to_pickle('Dataset_final_NLP.pkl')

CPU times: total: 44.3 s
Wall time: 44.3 s


In [12]:
df.to_pickle('Dataset_final_NLP.pkl')

In [ ]:
%%time

filas = []
for idx, row in df.iterrows():
    script_dict = row['Script_Dict']
    genders_dict = row.get('Characters_Genders', {})

    for personaje, texto in script_dict.items():
        if not texto or not isinstance(texto, str) or texto.strip() == "":
            continue

        filas.append({
            'IMDb_ID': row['IMDb_ID'],
            'Title': row['Title'],
            'Award': row['Award'],
            'Character': personaje,
            'Gender': genders_dict.get(personaje, 'unknown'),
            'Words': len(texto.split()),
            'Vader_Pos': row['VaderSentiment_Pos'].get(personaje),
            'Vader_Neu': row['VaderSentiment_Neu'].get(personaje),
            'Vader_Neg': row['VaderSentiment_Neg'].get(personaje),
            'Vader_Compound': row['VaderSentiment_Compound'].get(personaje),
            'Vader_Label': row['VaderSentiment_Label'].get(personaje)
        })

df_vader = pd.DataFrame(filas)

# ---------- Guardar ----------
df_vader.to_pickle('Dataset_final_NLP_flatten.pkl')
df_vader.to_csv('Dataset_final_NLP_flatten.csv', index=False)

# ---------- Resumen por género ----------
resumen = df_vader.groupby('Gender')[['Vader_Pos', 'Vader_Neu', 'Vader_Neg', 'Vader_Compound']].mean().round(3)
print(resumen)

resumen_label = df_vader.groupby(['Gender', 'Vader_Label']).size().unstack(fill_value=0)
print(resumen_label)

{'ARAGORN': 0.169,
 'ARWEN': 0.092,
 'BILBO': 0.164,
 'DENETHOR': 0.081,
 'DÉAGOL': 0.0,
 'ELROND': 0.048,
 'FARAMIR': 0.067,
 'FIGWIT': 0.329,
 'FRODO': 0.07,
 'GALADRIEL': 0.052,
 'GAMLING': 0.098,
 'GANDALF': 0.113,
 'GATE GUARD': 0.0,
 'GIMLI': 0.162,
 'GOLLUM': 0.103,
 'GORBAG': 0.25,
 'GOTHMOG': 0.034,
 'GRIMBOLD': 0.0,
 'GUARD': 0.119,
 'HEAD ORC': 0.0,
 'IORLAS': 0.0,
 'KING OF THE DEAD': 0.0,
 'LEGOLAS': 0.057,
 'LIEUTENANT': 0.0,
 'MADRIL': 0.0,
 'MERRY': 0.102,
 'ORC 2': 0.0,
 'ORC COMMANDER': 0.0,
 'ORCS': 0.0,
 'PIPPIN': 0.153,
 'ROHAN MARSHALL': 0.0,
 'ROHAN SOLDIER': 0.0,
 'ROHIRRIM': 0.0,
 'SAM': 0.065,
 'SHAGRAT': 0.065,
 'SMÉAGOL': 0.166,
 'SOLDIER': 0.0,
 'SOLDIER 1': 0.0,
 'SOLDIER 2': 0.0,
 'THÉODEN': 0.061,
 'TREEBEARD': 0.038,
 'WITCH KING': 0.085,
 'ÉOMER': 0.12,
 'ÉOWYN': 0.211}